# TorchFeather on Kaggle T4 GPUs

Trains a scaled-down DeepSeek V3 (MLA attention + MoE, ~59M parameters) on streamed FineWeb text.

**Before running:** Settings → Accelerator **GPU T4 x2**, and Internet **On** (needs a phone-verified Kaggle account).
Add the `torchfeather-code` dataset as input. With 2 GPUs this runs DDP, FSDP and tensor parallel; with 1 GPU it runs single-device training.

In [ ]:
import glob, os, shutil, subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

# Locate the uploaded code (the dataset mount path differs between Kaggle versions) and copy it to a writable folder.
matches = glob.glob("/kaggle/input/**/torchfeather/trainer.py", recursive=True)
if not matches:  # dataset kept as an archive: unpack it
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        shutil.unpack_archive(z, "/kaggle/tmp_code")
    matches = glob.glob("/kaggle/tmp_code/**/torchfeather/trainer.py", recursive=True)
assert matches, "Add the torchfeather-code dataset as notebook input"
src_root = os.path.dirname(os.path.dirname(matches[0]))
WORK = "/kaggle/working/torchfeather"
if not os.path.exists(WORK):
    shutil.copytree(src_root, WORK)
os.chdir(WORK)
print("code copied from", src_root)

In [ ]:
import torch
print("preinstalled torch", torch.__version__)
major, minor = (int(v) for v in torch.__version__.split("+")[0].split(".")[:2])
if (major, minor) < (2, 11):
    # The project targets torch 2.11; CUDA 12.8 wheels still support the T4 (compute capability 7.5).
    !pip install -q "torch==2.11.*" --index-url https://download.pytorch.org/whl/cu128
!pip install -q torchdata loguru "datasets>=2.21.0" "tokenizers>=0.22.1"

In [ ]:
# Restart-free check: run in a fresh interpreter so a just-upgraded torch is picked up.
!python -c "import torch, torchdata, datasets, tokenizers, loguru; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"

In [ ]:
# Tokenizer only (a few MB) from the public deepseek-moe-16b-base repo.
!python asset_scripts/hf_asset_downloader.py --repo_id deepseek-ai/deepseek-moe-16b-base --assets tokenizer
!ls assets/hf/deepseek-moe-16b-base

In [ ]:
n_gpus = int(subprocess.run([sys.executable, "-c", "import torch; print(torch.cuda.device_count())"], capture_output=True, text=True).stdout.strip())
# Short comparison of strategies: ["kaggle_t4_ddp", "kaggle_t4_fsdp", "kaggle_t4_tp"]. Longer run with validation: ["kaggle_t4_ddp_2k"].
CONFIGS = ["kaggle_t4_ddp_2k"] if n_gpus >= 2 else ["kaggle_t4_single"]
NPROC = 2 if n_gpus >= 2 else 1
print("GPUs:", n_gpus, "-> configs:", CONFIGS)

results = {}
for cfg in CONFIGS:
    print(f"\n===== {cfg} =====", flush=True)
    env = dict(os.environ, TORCHFEATHER_CONFIG=cfg, WANDB_MODE="disabled")
    with open(f"train_{cfg}.log", "w") as log:
        proc = subprocess.Popen(
            ["torchrun", "--standalone", f"--nproc_per_node={NPROC}", "-m", "torchfeather.trainer"],
            env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for line in proc.stdout:
            log.write(line)
            if any(k in line for k in ("step:", "Error", "error", "Traceback", "parameters", "Training completed")):
                print(line.rstrip(), flush=True)
        results[cfg] = proc.wait()
print("\nexit codes:", results)

In [ ]:
import json
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
summary = []
for idx, cfg in enumerate(CONFIGS):
    files = sorted(glob.glob(f"outputs/{cfg}/metrics/*/metrics.jsonl"))
    if not files:
        print("no metrics for", cfg); continue
    rows = [json.loads(l) for l in open(files[-1])]
    train = [r for r in rows if "loss_metrics/global_avg_loss" in r]
    val = [r for r in rows if "validation/loss" in r]
    color = f"C{idx}"
    ax.plot([r["step"] for r in train], [r["loss_metrics/global_avg_loss"] for r in train], color=color, alpha=0.55, label=f"{cfg} train")
    if val:
        ax.plot([r["step"] for r in val], [r["validation/loss"] for r in val], "o-", color=color, label=f"{cfg} validation")
    tps = [r["throughput(tps)"] for r in train[1:]] or [float("nan")]
    summary.append((cfg, train[0]["loss_metrics/global_avg_loss"], train[-1]["loss_metrics/global_avg_loss"],
                    val[-1]["validation/loss"] if val else float("nan"), sum(tps) / len(tps),
                    max(r["memory/max_reserved(GiB)"] for r in train)))
    for r in val:
        print(f"{cfg} validation step {r['step']:5}: loss {r['validation/loss']:.4f}  perplexity {r['validation/perplexity']:,.1f}")
ax.set_xlabel("step"); ax.set_ylabel("loss (nats)"); ax.set_title("TorchFeather (59M DeepSeek V3 MoE) on Kaggle T4")
ax.legend(); ax.grid(alpha=0.3)
fig.savefig("/kaggle/working/loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n{'config':18} {'first train':>11} {'final train':>11} {'final val':>10} {'tokens/s/GPU':>13} {'peak mem GiB':>13}")
for cfg, l0, l1, v1, tps, mem in summary:
    print(f"{cfg:18} {l0:11.3f} {l1:11.3f} {v1:10.3f} {tps:13,.0f} {mem:13.2f}")

for cfg in CONFIGS:
    path = f"outputs/{cfg}/samples.txt"
    if os.path.exists(path):
        print(f"\n===== samples: {cfg} =====")
        print(open(path, encoding="utf-8").read())

In [ ]:
# Keep the output small: drop checkpoints, keep logs, metrics and the plot.
for d in glob.glob("outputs/*/checkpoint"):
    shutil.rmtree(d, ignore_errors=True)
for f in glob.glob("train_*.log") + glob.glob("outputs/*/metrics/*/metrics.jsonl") + glob.glob("outputs/*/samples.txt"):
    dst = os.path.join("/kaggle/working/results", f.replace("/", "__"))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(f, dst)
print(os.listdir("/kaggle/working/results"))